# DL 시퀀스 데이터 준비
Purpose: build deterministic causal sequence indexes from the verified Goal 1.5 ML role views.

> Warning: this is an oracle/sanity-only synthetic-data benchmark, not real-device or medical-performance evidence.

In [ ]:
from __future__ import annotations

import hashlib
import json
import re
import sqlite3
import tempfile
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

SERIES_ID = "mvp3-oracle-v1"
EXPECTED_SPLIT_COUNTS = {"train": 24, "validation": 6, "locked_test": 6}
DATA_STATUS = "oracle/sanity"
REAL_ACCURACY_STATUS = "NOT VERIFIED"
DEVICE_SYNCHRONIZATION_STATUS = "NOT_AVAILABLE_TRUTH_ONLY"
RUN_TRAINING = False
RUN_LOCKED_TEST = False
SEQUENCE_LENGTHS_SECONDS = (300, 600)
SEQUENCE_OUTPUT_ROOT = Path("/kaggle/working/goal15_dl_sequences")
ML_VIEW_ROOT = Path("/kaggle/working/goal15_ml_view")
PATTERN_TARGET = "pattern_binary"
ONSET_EVENT_TARGET = "event_binary"
STAGE_TARGET = "stage_code"
STAGE_CODES = ("NO_EVENT", "LOW", "MEDIUM", "HIGH", "DECREASING", "RECOVERY")
BEHAVIOR_CODES = (
    "ear_covering", "exit_attempt", "head_turn_away", "motion_freeze",
    "movement_reduction", "repetitive_body_movement",
    "repetitive_hand_movement", "repetitive_object_contact",
    "sustained_pressure_or_contact", "withdrawal_movement",
)
CAUSAL_FACTORS = (
    "autonomic_arousal", "motor_activation", "cognitive_load", "sleep_pressure",
    "sensory_context", "recovery_capacity", "social_context",
)
ROLLING_STATISTICS = ("mean", "std", "slope")
ROLLING_WINDOWS_SECONDS = (5, 15, 30, 60, 180, 300)
TIME_FEATURE_COLUMNS = ("time_sin", "time_cos", "weekday_sin", "weekday_cos", "is_awake")
CONTEXT_FEATURE_COLUMNS = (
    "context__sleep", "context__transition", "context__meal_context",
    "context__focused_task", "context__moderate_activity",
    "context__light_activity", "context__wake_rest",
    "context__sedentary_activity",
)
ALLOWED_FEATURE_COLUMNS = tuple(
    [
        feature
        for factor in CAUSAL_FACTORS
        for feature in (
            f"{factor}__robust_z",
            *(
                f"{factor}__{statistic}_{window_seconds}s"
                for window_seconds in ROLLING_WINDOWS_SECONDS
                for statistic in ROLLING_STATISTICS
            ),
        )
    ]
    + list(TIME_FEATURE_COLUMNS)
    + list(CONTEXT_FEATURE_COLUMNS)
)
SEQUENCE_IDENTITY_COLUMNS = ("person_key", "run_id", "dataset_id", "canonical_time", "context")
APPROVED_CONTEXTS = ("sleep", "transition", "meal_context", "focused_task", "moderate_activity", "light_activity", "wake_rest", "sedentary_activity")
SEQUENCE_LABEL_COLUMNS = (PATTERN_TARGET, ONSET_EVENT_TARGET, "hard_negative", STAGE_TARGET, *BEHAVIOR_CODES)
SEQUENCE_METADATA_COLUMNS = frozenset({
    *SEQUENCE_IDENTITY_COLUMNS, "person_id", "split_role", "event_id", "session_id",
    "day_key", "day", "date", "missing_block", "is_missing_block",
    *SEQUENCE_LABEL_COLUMNS,
})
SEQUENCE_INDEX_COLUMNS = (
    "person_key", "run_id", "dataset_id", "context", "split_role",
    PATTERN_TARGET, ONSET_EVENT_TARGET, "hard_negative", STAGE_TARGET, *BEHAVIOR_CODES,
    "window_start", "window_end", "prediction_time", "length_seconds", "window_id", "sample_type",
)


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _require_sha256(value: Any, field: str) -> str:
    if not isinstance(value, str) or re.fullmatch(r"[0-9a-f]{64}", value) is None:
        raise ValueError(f"invalid {field} hash")
    return value


def derive_flat_dataset_identity(dataset_root: Path) -> tuple[str, str]:
    manifest_names = ("prepared__manifest.json", "outcomes__manifest.json", "registry__manifest.json")
    manifest_hashes: dict[str, str] = {}
    for name in manifest_names:
        path = dataset_root / name
        if not path.is_file():
            raise FileNotFoundError(f"missing original Dataset manifest: {name}")
        json.loads(path.read_text())
        manifest_hashes[name] = sha256_file(path)
    split_path = dataset_root / "registry__splits.parquet"
    if not split_path.is_file():
        raise FileNotFoundError("missing original Dataset split registry")
    source_dataset_hash = hashlib.sha256(json.dumps(manifest_hashes, sort_keys=True).encode()).hexdigest()
    return source_dataset_hash, sha256_file(split_path)


def validate_shared_dataset_identity(
    manifest: Mapping[str, Any], dataset_root: Path
) -> tuple[str, str]:
    if manifest.get("series_id") != SERIES_ID:
        raise ValueError("unexpected series identity")
    if manifest.get("data_status") != DATA_STATUS:
        raise ValueError("unexpected data status")
    declared_source = _require_sha256(manifest.get("source_dataset_hash"), "source dataset")
    declared_split = _require_sha256(manifest.get("split_hash"), "split")
    source_dataset_hash, split_hash = derive_flat_dataset_identity(dataset_root)
    if declared_source != source_dataset_hash:
        raise ValueError("source dataset hash does not match original flat Dataset")
    if declared_split != split_hash:
        raise ValueError("split hash does not match original flat Dataset")
    return source_dataset_hash, split_hash


def fit_train_normalization(
    frame: pd.DataFrame, *, feature_columns: Sequence[str], source_hash: str
) -> dict[str, Any]:
    _require_sha256(source_hash, "source")
    if "split_role" not in frame:
        raise ValueError("normalization requires split_role")
    missing = sorted(set(feature_columns).difference(frame.columns))
    if missing:
        raise ValueError(f"normalization features missing: {missing}")
    train = frame.loc[frame["split_role"].eq("train"), list(feature_columns)]
    if train.empty:
        raise ValueError("normalization requires train rows")
    statistics: dict[str, dict[str, float]] = {}
    for feature in feature_columns:
        values = pd.to_numeric(train[feature], errors="raise").to_numpy(dtype=np.float64)
        if not np.isfinite(values).all():
            raise ValueError(f"non-finite train feature: {feature}")
        median = float(np.median(values))
        lower, upper = np.quantile(values, [0.25, 0.75])
        iqr = float(upper - lower)
        statistics[feature] = {"median": median, "iqr": iqr if iqr > 0 else 1.0}
    return {
        "series_id": SERIES_ID,
        "fit_split_role": "train",
        "source_hash": source_hash,
        "features": statistics,
    }


def _require_non_null(frame: pd.DataFrame, columns: Sequence[str]) -> None:
    missing = sorted(set(columns).difference(frame.columns))
    if missing:
        raise ValueError(f"sequence contract missing columns: {missing}")
    null_columns = [column for column in columns if frame[column].isna().any()]
    if null_columns:
        raise ValueError(f"sequence contract has null columns: {null_columns}")


def _require_identity_text_and_context(frame: pd.DataFrame) -> None:
    text_columns = ("person_key", "run_id", "dataset_id", "context")
    _require_non_null(frame, text_columns)
    for column in text_columns:
        invalid = frame[column].map(lambda value: not isinstance(value, str) or not value or value != value.strip())
        if invalid.any():
            raise ValueError(f"invalid identity text: {column}")
    if not frame["context"].isin(APPROVED_CONTEXTS).all():
        raise ValueError("invalid context domain")


def _require_binary(frame: pd.DataFrame, columns: Sequence[str]) -> None:
    _require_non_null(frame, columns)
    invalid = [column for column in columns if not frame[column].isin((0, 1)).all()]
    if invalid:
        raise ValueError(f"sequence contract requires binary labels: {invalid}")


def validate_sequence_role_frame(frame: pd.DataFrame, split_role: str) -> list[str]:
    if split_role not in EXPECTED_SPLIT_COUNTS:
        raise ValueError(f"unknown split role: {split_role}")
    _require_non_null(frame, SEQUENCE_IDENTITY_COLUMNS)
    _require_identity_text_and_context(frame)
    _require_binary(frame, (PATTERN_TARGET, ONSET_EVENT_TARGET, "hard_negative", *BEHAVIOR_CODES))
    if frame[STAGE_TARGET].isna().any() or not frame[STAGE_TARGET].isin(STAGE_CODES).all():
        raise ValueError("sequence contract has invalid stage_code")
    expected_pattern = frame[STAGE_TARGET].ne("NO_EVENT").astype("int8")
    if not frame[PATTERN_TARGET].astype("int8").eq(expected_pattern).all():
        raise ValueError("pattern_binary must equal stage_code != NO_EVENT")
    if (frame[PATTERN_TARGET].eq(1) & frame["hard_negative"].eq(1)).any():
        raise ValueError("pattern and hard_negative cannot overlap")
    behavior_positive = frame.loc[:, list(BEHAVIOR_CODES)].eq(1).any(axis=1)
    ordinary_behavior = behavior_positive & frame[PATTERN_TARGET].eq(0) & frame["hard_negative"].eq(0)
    if ordinary_behavior.any():
        raise ValueError("behavior-positive ordinary baseline is not allowed")
    missing_features = [feature for feature in ALLOWED_FEATURE_COLUMNS if feature not in frame]
    if missing_features:
        raise ValueError(f"missing approved features: {missing_features}")
    unexpected = sorted(set(frame.columns).difference(ALLOWED_FEATURE_COLUMNS).difference(SEQUENCE_METADATA_COLUMNS))
    if unexpected:
        raise ValueError(f"unapproved DL sequence columns: {unexpected}")
    for feature in ALLOWED_FEATURE_COLUMNS:
        if not (pd.api.types.is_numeric_dtype(frame[feature]) or pd.api.types.is_bool_dtype(frame[feature])):
            raise ValueError(f"approved feature must be numeric or bool: {feature}")
        if not np.isfinite(frame[feature].to_numpy(dtype=np.float64)).all():
            raise ValueError(f"non-finite approved feature: {feature}")
    return list(ALLOWED_FEATURE_COLUMNS)


def _window_group_keys(frame: pd.DataFrame) -> list[str]:
    required = [*SEQUENCE_IDENTITY_COLUMNS, "split_role", PATTERN_TARGET, "hard_negative"]
    missing = sorted(set(required).difference(frame.columns))
    if missing:
        raise ValueError(f"sequence source missing columns: {missing}")
    group_keys = ["person_key", "run_id", "dataset_id", "context"]
    group_keys.extend(key for key in ("day_key", "day", "date", "session_id") if key in frame)
    return group_keys


def deterministic_window_id(endpoint: Mapping[str, Any], length_seconds: int) -> str:
    identity = (endpoint["person_key"], endpoint["run_id"], endpoint["dataset_id"], endpoint["context"], pd.Timestamp(endpoint["canonical_time"]).isoformat(), length_seconds)
    return hashlib.sha256("|".join(str(value) for value in identity).encode()).hexdigest()


def assert_window_boundaries(index: pd.DataFrame) -> None:
    required = {"person_key", "run_id", "dataset_id", "context", "window_start", "window_end", "prediction_time", "length_seconds", "window_id"}
    missing = sorted(required.difference(index.columns))
    if missing:
        raise ValueError(f"sequence index missing columns: {missing}")
    _require_identity_text_and_context(index)
    _require_non_null(index, ["window_start", "window_end", "prediction_time", "window_id"])
    if not index["length_seconds"].isin(SEQUENCE_LENGTHS_SECONDS).all():
        raise ValueError("sequence index has invalid length_seconds")
    for column in ("window_start", "window_end", "prediction_time"):
        if not isinstance(index[column].dtype, pd.DatetimeTZDtype) or str(index[column].dtype.tz) != "UTC":
            raise ValueError(f"{column} must be timezone-aware UTC")
    # Every candidate must satisfy window_end >= window_start.
    if not (index["window_end"] >= index["window_start"]).all():
        raise ValueError("window_end must be after window_start")
    expected_end = index["prediction_time"]
    if not index["window_end"].eq(expected_end).all():
        raise ValueError("window_end must equal prediction_time")
    expected_start = expected_end - pd.to_timedelta(index["length_seconds"] - 1, unit="s")
    if not index["window_start"].eq(expected_start).all():
        raise ValueError("causal window start mismatch")
    for _, group in index.groupby(["person_key", "run_id", "dataset_id", "context", "length_seconds"], sort=False):
        if group["prediction_time"].duplicated().any() or not group["prediction_time"].is_monotonic_increasing:
            raise ValueError("prediction_time must be unique and ordered")


def sequence_index_arrow_schema() -> pa.Schema:
    fields = [
        pa.field("person_key", pa.string(), nullable=False),
        pa.field("run_id", pa.string(), nullable=False),
        pa.field("dataset_id", pa.string(), nullable=False),
        pa.field("context", pa.string(), nullable=False),
        pa.field("split_role", pa.string(), nullable=False),
        pa.field(PATTERN_TARGET, pa.int8(), nullable=False),
        pa.field(ONSET_EVENT_TARGET, pa.int8()),
        pa.field("hard_negative", pa.int8(), nullable=False),
        pa.field(STAGE_TARGET, pa.string()),
    ]
    fields.extend(pa.field(code, pa.int8()) for code in BEHAVIOR_CODES)
    fields.extend([
        pa.field("window_start", pa.timestamp("ns", tz="UTC"), nullable=False),
        pa.field("window_end", pa.timestamp("ns", tz="UTC"), nullable=False),
        pa.field("prediction_time", pa.timestamp("ns", tz="UTC"), nullable=False),
        pa.field("length_seconds", pa.int32(), nullable=False),
        pa.field("window_id", pa.string(), nullable=False),
        pa.field("sample_type", pa.string(), nullable=False),
    ])
    return pa.schema(fields)


In [ ]:
def make_causal_window_index(frame: pd.DataFrame, *, length_seconds: int) -> pd.DataFrame:
    if isinstance(length_seconds, bool) or length_seconds not in SEQUENCE_LENGTHS_SECONDS:
        raise ValueError(f"length_seconds must be one of {SEQUENCE_LENGTHS_SECONDS}")
    group_keys = _window_group_keys(frame)
    work = frame.copy()
    _require_non_null(work, ["person_key", "run_id", "dataset_id", "context", "canonical_time", "split_role"])
    _require_identity_text_and_context(work)
    _require_binary(work, (PATTERN_TARGET, "hard_negative"))
    source_times = pd.to_datetime(work["canonical_time"], errors="raise")
    if not isinstance(source_times.dtype, pd.DatetimeTZDtype) or str(source_times.dtype.tz) != "UTC":
        raise ValueError("canonical_time must be timezone-aware UTC")
    work["canonical_time"] = source_times
    missing_column = next((name for name in ("missing_block", "is_missing_block") if name in work), None)
    records: list[dict[str, Any]] = []
    for _, group in work.groupby(group_keys, sort=False, dropna=False):
        ordered = group.sort_values("canonical_time", kind="mergesort").reset_index(drop=True)
        if ordered["canonical_time"].duplicated().any():
            raise ValueError("duplicate canonical_time within sequence group")
        for end_index in range(length_seconds - 1, len(ordered)):
            window = ordered.iloc[end_index - length_seconds + 1 : end_index + 1]
            prediction_time = window["canonical_time"].iloc[-1]
            window_start = prediction_time - pd.Timedelta(seconds=length_seconds - 1)
            window_end = prediction_time
            consecutive = window["canonical_time"].diff().dropna().eq(pd.Timedelta(seconds=1)).all()
            has_missing_block = bool(window[missing_column].fillna(True).astype(bool).any()) if missing_column else False
            if not consecutive or has_missing_block:
                continue
            endpoint = window.iloc[-1]
            record = {column: endpoint[column] for column in ("person_key", "run_id", "dataset_id", "context", "split_role", PATTERN_TARGET, "hard_negative")}
            record.update({
                ONSET_EVENT_TARGET: endpoint.get(ONSET_EVENT_TARGET),
                STAGE_TARGET: endpoint.get(STAGE_TARGET),
                **{code: endpoint.get(code, 0) for code in BEHAVIOR_CODES},
                "window_start": window_start,
                "window_end": window_end,
                "prediction_time": prediction_time,
                "length_seconds": length_seconds,
                "sample_type": "sliding",
            })
            record["window_id"] = deterministic_window_id(endpoint, length_seconds)
            records.append(record)
    index = pd.DataFrame(records, columns=list(SEQUENCE_INDEX_COLUMNS))
    index = index.sort_values(["person_key", "run_id", "dataset_id", "length_seconds", "prediction_time", "context"], kind="mergesort").reset_index(drop=True)
    if not index.empty:
        assert_window_boundaries(index)
        if index["window_id"].isna().any() or not index["window_id"].map(lambda value: isinstance(value, str) and bool(value)).all() or not index["window_id"].is_unique:
            raise ValueError("window_id must be unique non-empty text")
    return index


def sample_training_windows(index: pd.DataFrame, *, baseline_multiplier: int = 3) -> pd.DataFrame:
    if baseline_multiplier < 0:
        raise ValueError("baseline_multiplier must be non-negative")
    required = {"person_key", "run_id", "dataset_id", "context", "split_role", PATTERN_TARGET, "hard_negative", "length_seconds", "prediction_time", "window_id"}
    missing = sorted(required.difference(index.columns))
    if missing:
        raise ValueError(f"training index missing columns: {missing}")
    if not index["split_role"].eq("train").all():
        raise ValueError("training sampler accepts train windows only")
    _require_non_null(index, ["person_key", "run_id", "dataset_id", "context", "length_seconds", "prediction_time", "window_id"])
    _require_binary(index, (PATTERN_TARGET, "hard_negative"))
    positive = index[PATTERN_TARGET].eq(1)
    hard_negative = ~positive & index["hard_negative"].eq(1)
    baseline = ~(positive | hard_negative)
    selected = [
        index.loc[positive].assign(sample_type="positive_centered"),
        index.loc[hard_negative].assign(sample_type="hard_negative"),
    ]
    strata = ["person_key", "run_id", "context", "length_seconds"]
    baseline_rows: list[pd.DataFrame] = []
    for _, positives in index.loc[positive].groupby(strata, sort=False, dropna=False):
        key = tuple(positives.iloc[0][column] for column in strata)
        matched = index.loc[baseline].copy()
        for column, value in zip(strata, key, strict=True):
            matched = matched.loc[matched[column].eq(value)]
        matched["_sample_hash"] = pd.util.hash_pandas_object(
            matched[["person_key", "run_id", "dataset_id", "context", "prediction_time", "window_id"]],
            index=False, categorize=True,
        )
        baseline_rows.append(
            matched.sort_values(["_sample_hash", "window_id"], kind="mergesort")
            .head(baseline_multiplier * len(positives))
            .drop(columns="_sample_hash")
            .assign(sample_type="matched_baseline")
        )
    selected.extend(baseline_rows)
    return pd.concat(selected, ignore_index=True).sort_values(
        ["prediction_time", "window_id"], kind="mergesort"
    ).reset_index(drop=True)


def write_train_normalization(output_root: Path, statistics: Mapping[str, Any]) -> Path:
    output_root.mkdir(parents=True, exist_ok=True)
    path = output_root / "train_normalization.json"
    path.write_text(json.dumps(statistics, indent=2, sort_keys=True) + "\n")
    return path


def write_sequence_manifest(
    output_root: Path, *, index_paths: Mapping[str, Path], normalization_path: Path,
    source_dataset_hash: str, split_hash: str, row_counts: Mapping[str, int],
) -> Path:
    _require_sha256(source_dataset_hash, "source dataset")
    _require_sha256(split_hash, "split")
    if set(index_paths) != set(row_counts):
        raise ValueError("sequence manifest role metadata mismatch")
    if not normalization_path.is_file():
        raise FileNotFoundError("missing train normalization statistics")
    files: dict[str, dict[str, Any]] = {}
    for name, path in sorted(index_paths.items()):
        count = row_counts[name]
        if not path.is_file() or isinstance(count, bool) or not isinstance(count, int) or count < 0:
            raise ValueError(f"invalid sequence output: {name}")
        files[name] = {"path": path.name, "sha256": sha256_file(path), "row_count": count}
    manifest_path = output_root / "sequence_manifest.json"
    manifest_path.write_text(json.dumps({
        "series_id": SERIES_ID, "data_status": DATA_STATUS,
        "source_dataset_hash": source_dataset_hash, "split_hash": split_hash,
        "normalization": {"path": normalization_path.name, "sha256": sha256_file(normalization_path)},
        "files": files,
    }, indent=2, sort_keys=True) + "\n")
    return manifest_path


In [ ]:
def resolve_ml_view_root(input_root: Path = Path("/kaggle/input")) -> Path:
    children = sorted(path for path in input_root.iterdir() if path.is_dir()) if input_root.exists() else []
    for candidate in [ML_VIEW_ROOT, input_root, *children]:
        if (candidate / "view_manifest.json").is_file():
            return candidate
    raise FileNotFoundError("verified goal15_ml_view manifest is required")


def resolve_flat_dataset_root(input_root: Path = Path("/kaggle/input")) -> Path:
    children = sorted(path for path in input_root.iterdir() if path.is_dir()) if input_root.exists() else []
    required = {"prepared__manifest.json", "outcomes__manifest.json", "registry__manifest.json", "registry__splits.parquet"}
    for candidate in [input_root, *children]:
        if required.issubset({path.name for path in candidate.iterdir()}):
            return candidate
    raise FileNotFoundError("original flat Goal 1.5 Dataset is required")


def _flat_split_membership(dataset_root: Path) -> dict[str, set[str]]:
    split = pq.read_table(dataset_root / "registry__splits.parquet", columns=["person_key", "split_role"]).to_pandas()
    _require_non_null(split, ["person_key", "split_role"])
    if not split["split_role"].isin(EXPECTED_SPLIT_COUNTS).all():
        raise ValueError("original split has invalid role")
    memberships = {role: set(split.loc[split["split_role"].eq(role), "person_key"]) for role in EXPECTED_SPLIT_COUNTS}
    if {role: len(memberships[role]) for role in EXPECTED_SPLIT_COUNTS} != EXPECTED_SPLIT_COUNTS:
        raise ValueError("original split is not 24/6/6")
    if any(memberships[left] & memberships[right] for left in memberships for right in memberships if left < right):
        raise ValueError("original split has person overlap")
    return memberships


def _verified_role_views(root: Path, dataset_root: Path) -> tuple[dict[str, Path], str, str]:
    manifest = json.loads((root / "view_manifest.json").read_text())
    source_dataset_hash, split_hash = validate_shared_dataset_identity(manifest, dataset_root)
    files = manifest.get("files")
    if not isinstance(files, dict) or set(files) != set(EXPECTED_SPLIT_COUNTS):
        raise ValueError("ML view manifest must declare all immutable split roles")
    expected_membership = _flat_split_membership(dataset_root)
    paths: dict[str, Path] = {}
    expected_feature_types: dict[str, pa.DataType] | None = None
    for split_role, metadata in files.items():
        path = root / metadata["path"]
        if not path.is_file() or sha256_file(path) != _require_sha256(metadata.get("sha256"), split_role):
            raise ValueError(f"unverified ML role view: {split_role}")
        parquet = pq.ParquetFile(path)
        columns = list(parquet.schema_arrow.names)
        if metadata.get("columns") != columns:
            raise ValueError(f"ML role view schema metadata mismatch: {split_role}")
        missing_features = [feature for feature in ALLOWED_FEATURE_COLUMNS if feature not in columns]
        if missing_features:
            raise ValueError(f"missing approved features: {missing_features}")
        feature_types = {feature: parquet.schema_arrow.field(feature).type for feature in ALLOWED_FEATURE_COLUMNS}
        if expected_feature_types is None:
            expected_feature_types = feature_types
        elif feature_types != expected_feature_types:
            raise ValueError(f"feature schema mismatch: {split_role}")
        people: set[str] = set()
        for row_group in range(parquet.num_row_groups):
            person_table = parquet.read_row_group(row_group, columns=["person_key"])
            person_values = person_table.column("person_key").to_pandas()
            if person_values.isna().any():
                raise ValueError("ML role view has null person_key")
            people.update(person_values.tolist())
        if people != expected_membership[split_role]:
            raise ValueError(f"ML role view membership mismatch: {split_role}")
        paths[split_role] = path
    return paths, source_dataset_hash, split_hash


def _row_group_identity_columns(frame: pd.DataFrame) -> list[str]:
    columns = ["person_key", "run_id", "dataset_id"]
    columns.extend(column for column in ("day_key", "day", "date", "session_id") if column in frame)
    return columns


def _iter_role_person_chunks(path: Path, split_role: str):
    parquet = pq.ParquetFile(path)
    previous_identity: tuple[Any, ...] | None = None
    previous_time: pd.Timestamp | None = None
    previous_source_identity: tuple[Any, ...] | None = None
    previous_source_time: pd.Timestamp | None = None
    for row_group in range(parquet.num_row_groups):
        frame = parquet.read_row_group(row_group).to_pandas()
        if "split_role" not in frame:
            frame["split_role"] = split_role
        elif not frame["split_role"].eq(split_role).all():
            raise ValueError(f"role column mismatch: {split_role}")
        validate_sequence_role_frame(frame, split_role)
        identity_columns = _row_group_identity_columns(frame)
        if any(frame[column].nunique(dropna=False) != 1 for column in identity_columns):
            raise ValueError("ML role row group must contain one ordered sequence identity")
        identity = tuple(frame.iloc[0][column] for column in identity_columns)
        times = pd.to_datetime(frame["canonical_time"], errors="raise")
        if not isinstance(times.dtype, pd.DatetimeTZDtype) or str(times.dtype.tz) != "UTC":
            raise ValueError("ML role row group canonical_time must be timezone-aware UTC")
        if times.duplicated().any() or not times.is_monotonic_increasing:
            raise ValueError("ML role row group timestamps must be unique and ordered")
        first_time, last_time = pd.Timestamp(times.iloc[0]), pd.Timestamp(times.iloc[-1])
        if previous_identity is not None:
            if identity < previous_identity:
                raise ValueError("ML role row groups must be contiguous and globally ordered")
            if identity == previous_identity and first_time <= previous_time:
                raise ValueError("ML role row group endpoints must be unique and ordered")
        source_identity = tuple(frame.iloc[0][column] for column in ("person_key", "run_id", "dataset_id"))
        if previous_source_identity == source_identity and first_time <= previous_source_time:
            raise ValueError("ML role source endpoints must be unique and ordered")
        previous_identity, previous_time = identity, last_time
        previous_source_identity, previous_source_time = source_identity, last_time
        yield frame


def _combine_contiguous_tail(tail: pd.DataFrame, current: pd.DataFrame, *, max_length: int) -> pd.DataFrame:
    if current.empty:
        return tail.copy()
    if tail.empty:
        return current.copy()
    identity = ["person_key", "run_id", "dataset_id", "context"]
    identity.extend(column for column in ("day_key", "day", "date", "session_id") if column in tail and column in current)
    same_identity = all(tail.iloc[-1][column] == current.iloc[0][column] for column in identity)
    previous_time = pd.Timestamp(tail.iloc[-1]["canonical_time"])
    current_time = pd.Timestamp(current.iloc[0]["canonical_time"])
    if not same_identity or current_time - previous_time != pd.Timedelta(seconds=1):
        return current.copy()
    return pd.concat([tail, current], ignore_index=True)


def _index_current_chunk(tail: pd.DataFrame, current: pd.DataFrame, *, length_seconds: int) -> pd.DataFrame:
    combined = _combine_contiguous_tail(tail, current, max_length=max(SEQUENCE_LENGTHS_SECONDS))
    index = make_causal_window_index(combined, length_seconds=length_seconds)
    if current.empty:
        return index.iloc[0:0].copy()
    first_current_time = pd.Timestamp(current["canonical_time"].iloc[0])
    return index.loc[index["prediction_time"].ge(first_current_time)].reset_index(drop=True)


def _carry_tail_after_boundaries(frame: pd.DataFrame, *, max_length: int) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    times = pd.to_datetime(frame["canonical_time"], errors="raise")
    reset_at = 0
    discontinuity = times.diff().ne(pd.Timedelta(seconds=1))
    discontinuity.iloc[0] = False
    if discontinuity.any():
        reset_at = int(np.flatnonzero(discontinuity.to_numpy())[-1])
    for column in ("missing_block", "is_missing_block"):
        if column in frame:
            missing = frame[column].fillna(True).astype(bool).to_numpy()
            if missing.any():
                reset_at = max(reset_at, int(np.flatnonzero(missing)[-1]) + 1)
    return frame.iloc[reset_at:].tail(max_length - 1).copy()


def _validate_index_append(index: pd.DataFrame, state: Mapping[str, Any] | None = None) -> dict[str, Any]:
    next_state = dict(state or {})
    if index.empty:
        return next_state
    assert_window_boundaries(index)
    last_order = next_state.get("last_order")
    last_window_id = next_state.get("last_window_id")
    for row in index.itertuples(index=False):
        endpoint = {"person_key": row.person_key, "run_id": row.run_id, "dataset_id": row.dataset_id, "context": row.context, "canonical_time": row.prediction_time}
        expected_id = deterministic_window_id(endpoint, int(row.length_seconds))
        if not isinstance(row.window_id, str) or not row.window_id or row.window_id != expected_id:
            raise ValueError("sequence index has invalid deterministic window_id")
        order = (row.person_key, row.run_id, row.dataset_id, int(row.length_seconds), pd.Timestamp(row.prediction_time).value, row.context)
        if last_order is not None and order <= last_order:
            raise ValueError("sequence index endpoints must be globally ordered and unique")
        if last_window_id is not None and row.window_id == last_window_id:
            raise ValueError("sequence index window_id must be unique")
        last_order, last_window_id = order, row.window_id
    next_state.update({"last_order": last_order, "last_window_id": last_window_id})
    return next_state


def _baseline_rank(window_id: str) -> str:
    return hashlib.sha256(window_id.encode()).hexdigest()


def _sample_train_index_file(raw_path: Path, output_path: Path, *, temporary_directory: Path) -> int:
    with tempfile.NamedTemporaryFile(prefix="goal15-train-sampling-", suffix=".sqlite", dir=temporary_directory, delete=False) as handle:
        database_path = Path(handle.name)
    connection = sqlite3.connect(database_path)
    writer: pq.ParquetWriter | None = None
    row_count = 0
    raw_state: dict[str, Any] = {}
    output_state: dict[str, Any] = {}
    strata = ("person_key", "run_id", "context", "length_seconds")
    try:
        connection.executescript("""
            CREATE TABLE positives (person_key TEXT, run_id TEXT, context TEXT, length_seconds INTEGER, positive_count INTEGER NOT NULL, PRIMARY KEY (person_key, run_id, context, length_seconds));
            CREATE TABLE baseline_candidates (person_key TEXT, run_id TEXT, context TEXT, length_seconds INTEGER, sample_rank TEXT NOT NULL, window_id TEXT PRIMARY KEY);
        """)
        parquet = pq.ParquetFile(raw_path)
        for row_group in range(parquet.num_row_groups):
            frame = parquet.read_row_group(row_group).to_pandas()
            raw_state = _validate_index_append(frame, raw_state)
            positive = frame[PATTERN_TARGET].eq(1)
            for key, group in frame.loc[positive].groupby(list(strata), sort=False, dropna=False):
                connection.execute("INSERT INTO positives VALUES (?, ?, ?, ?, ?) ON CONFLICT(person_key, run_id, context, length_seconds) DO UPDATE SET positive_count = positive_count + excluded.positive_count", (*key, len(group)))
            baseline = frame.loc[~positive & frame["hard_negative"].eq(0)]
            connection.executemany("INSERT INTO baseline_candidates VALUES (?, ?, ?, ?, ?, ?)", [(*tuple(getattr(row, column) for column in strata), _baseline_rank(row.window_id), row.window_id) for row in baseline.itertuples(index=False)])
        connection.executescript("""
            CREATE TABLE chosen_baselines (window_id TEXT PRIMARY KEY);
            INSERT INTO chosen_baselines
            SELECT window_id FROM (
                SELECT baseline_candidates.window_id AS window_id, positives.positive_count AS positive_count,
                    ROW_NUMBER() OVER (PARTITION BY baseline_candidates.person_key, baseline_candidates.run_id, baseline_candidates.context, baseline_candidates.length_seconds ORDER BY baseline_candidates.sample_rank, baseline_candidates.window_id) AS baseline_rank
                FROM baseline_candidates JOIN positives USING (person_key, run_id, context, length_seconds)
            ) WHERE baseline_rank <= 3 * positive_count;
        """)
        for row_group in range(parquet.num_row_groups):
            frame = parquet.read_row_group(row_group).to_pandas()
            chosen_ids: set[str] = set()
            window_ids = frame["window_id"].tolist()
            for offset in range(0, len(window_ids), 500):
                batch = window_ids[offset : offset + 500]
                placeholders = ",".join("?" for _ in batch)
                chosen_ids.update(row[0] for row in connection.execute(f"SELECT window_id FROM chosen_baselines WHERE window_id IN ({placeholders})", batch))
            positive = frame[PATTERN_TARGET].eq(1)
            hard_negative = ~positive & frame["hard_negative"].eq(1)
            selected = frame.loc[positive | hard_negative | frame["window_id"].isin(chosen_ids)].copy()
            if selected.empty:
                continue
            selected["sample_type"] = np.where(selected[PATTERN_TARGET].eq(1), "positive_centered", np.where(selected["hard_negative"].eq(1), "hard_negative", "matched_baseline"))
            output_state = _validate_index_append(selected, output_state)
            table = _sequence_index_table(selected)
            if writer is None:
                writer = pq.ParquetWriter(output_path, table.schema, compression="zstd")
            writer.write_table(table)
            row_count += len(selected)
        if writer is None:
            empty = _sequence_index_table(pd.DataFrame())
            writer = pq.ParquetWriter(output_path, empty.schema, compression="zstd")
            writer.write_table(empty)
    finally:
        if writer is not None:
            writer.close()
        connection.close()
        database_path.unlink(missing_ok=True)
    return row_count


def fit_train_normalization_from_role_file(
    path: Path, *, feature_columns: Sequence[str], source_hash: str, temporary_directory: Path | None = None
) -> dict[str, Any]:
    _require_sha256(source_hash, "source")
    parquet = pq.ParquetFile(path)
    statistics: dict[str, dict[str, float]] = {}
    temporary_directory = temporary_directory or path.parent
    temporary_directory.mkdir(parents=True, exist_ok=True)
    def exact_memmap_quantile(values: np.memmap, quantile: float) -> float:
        position = (len(values) - 1) * quantile
        lower_index = int(np.floor(position))
        upper_index = int(np.ceil(position))
        values.partition((lower_index, upper_index))
        return float(values[lower_index] + (values[upper_index] - values[lower_index]) * (position - lower_index))
    for feature in feature_columns:
        row_count = sum(parquet.metadata.row_group(index).num_rows for index in range(parquet.num_row_groups))
        if row_count < 1:
            raise ValueError(f"normalization has no train values: {feature}")
        with tempfile.NamedTemporaryFile(prefix="goal15-normalization-", suffix=".mmap", dir=temporary_directory, delete=False) as handle:
            mmap_path = Path(handle.name)
        values: np.memmap | None = None
        try:
            values = np.memmap(mmap_path, mode="w+", dtype=np.float64, shape=(row_count,))
            offset = 0
            for row_group in range(parquet.num_row_groups):
                batch = parquet.read_row_group(row_group, columns=[feature]).column(feature).to_numpy(zero_copy_only=False)
                numeric = np.asarray(batch, dtype=np.float64)
                if not np.isfinite(numeric).all():
                    raise ValueError(f"non-finite train feature: {feature}")
                values[offset : offset + len(numeric)] = numeric
                offset += len(numeric)
            values.flush()
            lower = exact_memmap_quantile(values, 0.25)
            median = exact_memmap_quantile(values, 0.5)
            upper = exact_memmap_quantile(values, 0.75)
            iqr = float(upper - lower)
            statistics[feature] = {"median": float(median), "iqr": iqr if iqr > 0 else 1.0}
        finally:
            if values is not None:
                values.flush()
                del values
            mmap_path.unlink(missing_ok=True)
    return {"series_id": SERIES_ID, "fit_split_role": "train", "source_hash": source_hash, "features": statistics}


def _sequence_index_table(index: pd.DataFrame) -> pa.Table:
    schema = sequence_index_arrow_schema()
    if index.empty:
        return pa.Table.from_batches([], schema=schema)
    return pa.Table.from_pandas(index.loc[:, list(SEQUENCE_INDEX_COLUMNS)], schema=schema, preserve_index=False)


def build_all_sequence_indexes(
    *, flat_dataset_root: Path | None = None, ml_view_root: Path | None = None, output_root: Path = SEQUENCE_OUTPUT_ROOT
) -> Path:
    dataset_root = flat_dataset_root or resolve_flat_dataset_root()
    views_root = ml_view_root or resolve_ml_view_root()
    role_paths, source_dataset_hash, split_hash = _verified_role_views(views_root, dataset_root)
    for split_role, path in role_paths.items():
        for _ in _iter_role_person_chunks(path, split_role):
            pass
    statistics = fit_train_normalization_from_role_file(
        role_paths["train"], feature_columns=ALLOWED_FEATURE_COLUMNS, source_hash=source_dataset_hash
    )
    normalization_path = write_train_normalization(output_root, statistics)
    output_root.mkdir(parents=True, exist_ok=True)
    expected_names = {f"{role}_{length}" for role in EXPECTED_SPLIT_COUNTS for length in SEQUENCE_LENGTHS_SECONDS}
    index_paths = {name: output_root / f"{name}.parquet" for name in expected_names}
    row_counts = {name: 0 for name in expected_names}
    raw_paths: dict[str, Path] = {}
    for name in expected_names:
        if name.startswith("train_"):
            with tempfile.NamedTemporaryFile(prefix=f"goal15-{name}-", suffix=".parquet", dir=output_root, delete=False) as handle:
                raw_paths[name] = Path(handle.name)
        else:
            raw_paths[name] = index_paths[name]
    writers: dict[str, pq.ParquetWriter] = {}
    append_states: dict[str, dict[str, Any]] = {name: {} for name in expected_names}
    try:
        for split_role, path in role_paths.items():
            tail = pd.DataFrame()
            for frame in _iter_role_person_chunks(path, split_role):
                combined = _combine_contiguous_tail(tail, frame, max_length=max(SEQUENCE_LENGTHS_SECONDS))
                for length_seconds in SEQUENCE_LENGTHS_SECONDS:
                    index = make_causal_window_index(combined, length_seconds=length_seconds)
                    first_current_time = pd.Timestamp(frame["canonical_time"].iloc[0])
                    index = index.loc[index["prediction_time"].ge(first_current_time)].reset_index(drop=True)
                    name = f"{split_role}_{length_seconds}"
                    table = _sequence_index_table(index)
                    if table.num_rows == 0:
                        continue
                    append_states[name] = _validate_index_append(index, append_states[name])
                    if name not in writers:
                        writers[name] = pq.ParquetWriter(raw_paths[name], table.schema, compression="zstd")
                    writers[name].write_table(table)
                    if split_role != "train":
                        row_counts[name] += len(index)
                tail = _carry_tail_after_boundaries(combined, max_length=max(SEQUENCE_LENGTHS_SECONDS))
        for name in expected_names:
            if name not in writers:
                empty_table = _sequence_index_table(pd.DataFrame())
                writers[name] = pq.ParquetWriter(raw_paths[name], empty_table.schema, compression="zstd")
                writers[name].write_table(empty_table)
    finally:
        for writer in writers.values():
            writer.close()
    try:
        for length_seconds in SEQUENCE_LENGTHS_SECONDS:
            name = f"train_{length_seconds}"
            row_counts[name] = _sample_train_index_file(raw_paths[name], index_paths[name], temporary_directory=output_root)
    finally:
        for name, raw_path in raw_paths.items():
            if name.startswith("train_"):
                raw_path.unlink(missing_ok=True)
    if set(index_paths) != expected_names:
        raise ValueError("missing sequence index output")
    return write_sequence_manifest(
        output_root, index_paths=index_paths, normalization_path=normalization_path,
        source_dataset_hash=source_dataset_hash, split_hash=split_hash, row_counts=row_counts,
    )


In [ ]:
RUN_DATA_PREPARATION = False

if RUN_DATA_PREPARATION:
    manifest_path = build_all_sequence_indexes()
    print(f"시퀀스 인덱스 생성 완료: {manifest_path}")
else:
    print("시퀀스 데이터 준비 비활성화: RUN_DATA_PREPARATION=False")
